# Market Data Quality Report
**ATS-163 — Collect and align market data (prices, volumes, corporate actions)**

This notebook assesses the quality and completeness of the aligned market dataset produced by `build_market_dataset()`.  It is intended for a CDO-level review of:

- Universe coverage (tickers, exchanges, sectors, date ranges)
- Price and volume data completeness
- Missing-data patterns by sector and year
- Forward-return and market-adjusted-return distributions
- Corporate actions summary
- Stock-decline label preview (>30 % drop in 12 months)

---
*Run `scripts/run_market_pipeline.py` first to generate the data files.*

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Paths ─────────────────────────────────────────────────────────────────
REPO_ROOT   = Path().resolve().parent   # adjust if running from elsewhere
RAW_DIR     = REPO_ROOT / "data" / "raw"
MARKET_DIR  = RAW_DIR / "market"
UNIVERSE_PATH   = RAW_DIR / "company_universe.parquet"
ALIGNED_PATH    = MARKET_DIR / "market_aligned.parquet"
ACTIONS_DIR     = MARKET_DIR / "actions"
PRICES_DIR      = MARKET_DIR / "prices"

# ── Plot style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
PALETTE = sns.color_palette("husl", 12)

def _check_file(p: Path, label: str) -> bool:
    if p.exists():
        print(f"✓  {label}: {p}")
        return True
    print(f"✗  {label} NOT FOUND — run scripts/run_market_pipeline.py first")
    return False

_universe_ok = _check_file(UNIVERSE_PATH, "Company universe")
_aligned_ok  = _check_file(ALIGNED_PATH,  "Aligned market data")

---
## 1  Load Data

In [ ]:
# ── Universe ──────────────────────────────────────────────────────────────
if _universe_ok:
    from fin_jepa.data.universe import load_company_universe
    universe, provenance = load_company_universe(UNIVERSE_PATH)
    print(f"Universe: {len(universe):,} companies")
    print(f"Provenance: {provenance}")
else:
    universe = pd.DataFrame()

# ── Aligned market data ───────────────────────────────────────────────────
if _aligned_ok:
    mkt = pd.read_parquet(ALIGNED_PATH)
    mkt["filing_date"]  = pd.to_datetime(mkt["filing_date"])
    mkt["period_end"]   = pd.to_datetime(mkt["period_end"])
    mkt["filing_year"]  = mkt["filing_date"].dt.year
    mkt["period_year"]  = mkt["period_end"].dt.year
    print(f"\nAligned dataset: {len(mkt):,} rows × {len(mkt.columns)} columns")
    print(f"Columns: {list(mkt.columns)}")
    mkt.head(3)
else:
    mkt = pd.DataFrame()

---
## 2  Universe Overview

In [ ]:
if not universe.empty:
    print("=" * 50)
    print("COMPANY UNIVERSE SUMMARY")
    print("=" * 50)
    print(f"  Total companies          : {len(universe):,}")
    print(f"  With exchange ticker     : {universe['ticker'].notna().sum():,} "
          f"({universe['ticker'].notna().mean()*100:.1f}%)")
    print(f"  XBRL coverage ≥ 80%      : {(universe['xbrl_coverage_pct'] >= 0.8).sum():,}")
    print(f"  Active filers (is_current): {universe['is_current_filer'].sum():,}")
    if 'first_10k_date' in universe.columns:
        print(f"  Earliest 10-K            : {pd.to_datetime(universe['first_10k_date']).min().date()}")
        print(f"  Latest 10-K              : {pd.to_datetime(universe['last_10k_date']).max().date()}")
    print()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Sector distribution
    sec_counts = universe["sector"].value_counts()
    axes[0].barh(sec_counts.index, sec_counts.values, color=PALETTE)
    axes[0].set_title("Companies by Sector (FF-12)")
    axes[0].set_xlabel("Count")

    # Exchange distribution
    exch = universe["exchange"].fillna("Unknown").value_counts().head(8)
    axes[1].bar(exch.index, exch.values, color=PALETTE[:len(exch)])
    axes[1].set_title("Companies by Exchange")
    axes[1].set_xlabel("Exchange")
    axes[1].tick_params(axis='x', rotation=30)

    # XBRL coverage distribution
    axes[2].hist(
        universe["xbrl_coverage_pct"].dropna() * 100,
        bins=20, color=PALETTE[2], edgecolor="white"
    )
    axes[2].set_title("XBRL Coverage % per Company")
    axes[2].set_xlabel("XBRL Coverage (%)")
    axes[2].set_ylabel("# Companies")

    plt.tight_layout()
    plt.show()

In [ ]:
if not universe.empty:
    # Filings per year (how many companies filed a 10-K each year)
    year_counts = (
        universe["filing_years"]
        .explode()
        .value_counts()
        .sort_index()
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(year_counts.index.astype(str), year_counts.values, color=PALETTE[1])
    ax.set_title("10-K Filings per Calendar Year (Universe)")
    ax.set_xlabel("Year")
    ax.set_ylabel("# Companies with a 10-K")
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
    print(year_counts.to_frame("n_companies").T)

---
## 3  Market Data Coverage

In [ ]:
if not mkt.empty:
    print("=" * 55)
    print("ALIGNED DATASET COVERAGE SUMMARY")
    print("=" * 55)
    print(f"  Company-year rows        : {len(mkt):,}")
    print(f"  Unique CIKs              : {mkt['cik'].nunique():,}")
    print(f"  Unique tickers           : {mkt['ticker'].nunique():,}")
    print(f"  Filing date range        : {mkt['filing_date'].min().date()} → "
          f"{mkt['filing_date'].max().date()}")
    print(f"  Period-end range         : {mkt['period_end'].min().date()} → "
          f"{mkt['period_end'].max().date()}")
    print()
    print("  Return coverage:")
    for col in ["fwd_ret_21d", "fwd_ret_63d", "fwd_ret_126d", "fwd_ret_252d"]:
        if col in mkt.columns:
            pct = mkt[col].notna().mean() * 100
            print(f"    {col:<18}: {pct:5.1f}% non-null")
    print()
    delisted_pct = mkt["delisted"].mean() * 100 if "delisted" in mkt.columns else float("nan")
    print(f"  Delisted (< 252d after filing): {delisted_pct:.1f}%")

In [ ]:
if not mkt.empty:
    # Coverage by filing year and horizon
    horizons = [c for c in ["fwd_ret_21d", "fwd_ret_63d", "fwd_ret_126d", "fwd_ret_252d"]
                if c in mkt.columns]
    cov = (
        mkt.groupby("filing_year")[horizons]
        .apply(lambda df: df.notna().mean() * 100)
        .rename(columns={c: c.replace("fwd_ret_", "") for c in horizons})
    )

    fig, ax = plt.subplots(figsize=(13, 4))
    cov.plot(ax=ax, marker="o", linewidth=2)
    ax.set_title("Return Coverage (% non-null) by Filing Year and Horizon")
    ax.set_xlabel("Filing Year")
    ax.set_ylabel("Coverage (%)")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(title="Horizon", bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.show()

---
## 4  Missing Data Heatmap

In [ ]:
if not mkt.empty and "sector" in mkt.columns:
    # Missing fwd_ret_252d by sector × year
    pivot = (
        mkt.assign(missing_252d=(~mkt["fwd_ret_252d"].notna()).astype(int))
        .groupby(["sector", "filing_year"])["missing_252d"]
        .mean()
        .mul(100)
        .unstack("filing_year")
        .fillna(100.0)
    )

    fig, ax = plt.subplots(figsize=(16, 6))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="YlOrRd",
        annot=True,
        fmt=".0f",
        linewidths=0.3,
        cbar_kws={"label": "% Missing fwd_ret_252d"},
        vmin=0,
        vmax=100,
    )
    ax.set_title("Missing 12-Month Forward Return (%) by Sector × Filing Year",
                 fontsize=13, pad=12)
    ax.set_xlabel("Filing Year")
    ax.set_ylabel("Sector (FF-12)")
    plt.tight_layout()
    plt.show()

In [ ]:
if not mkt.empty:
    # All return columns — percent missing bar chart
    ret_cols = [c for c in mkt.columns if c.startswith("fwd_ret_") or c.startswith("mkt_adj_") or c.startswith("sec_adj_")]
    missing_pct = mkt[ret_cols].isna().mean().sort_values() * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#e74c3c" if v > 20 else "#2ecc71" for v in missing_pct]
    ax.barh(missing_pct.index, missing_pct.values, color=colors)
    ax.axvline(20, color="#e74c3c", linestyle="--", linewidth=1, label="20% threshold")
    ax.set_title("Missing Data by Return Column")
    ax.set_xlabel("% Missing")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(missing_pct.to_frame("pct_missing").to_string())

---
## 5  Price and Volume Distributions

In [ ]:
# Sample a few tickers and plot their price history
if PRICES_DIR.exists():
    parquet_files = sorted(PRICES_DIR.glob("*.parquet"))
    print(f"Cached price files: {len(parquet_files):,}")

    # Sample tickers from each sector if universe is available
    if not universe.empty and not mkt.empty:
        sample_tickers = (
            mkt[mkt["fwd_ret_252d"].notna()]
            .groupby("sector")["ticker"]
            .first()
            .dropna()
            .tolist()[:8]
        )
    else:
        sample_tickers = [f.stem for f in parquet_files[:8]]

    fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=False)
    for ax, ticker in zip(axes.flat, sample_tickers):
        safe = ticker.replace("^", "_idx_").replace("/", "_")
        path = PRICES_DIR / f"{safe}.parquet"
        if path.exists():
            df = pd.read_parquet(path)
            if not df.empty and "Close" in df.columns:
                df["Close"].plot(ax=ax, linewidth=0.8, color=PALETTE[2])
                ax.set_title(ticker, fontsize=10)
                ax.set_xlabel("")
                ax.tick_params(axis='x', rotation=30, labelsize=8)
        else:
            ax.set_visible(False)

    fig.suptitle("Sample Adjusted Close Prices (one per sector)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Price cache directory not found — run the pipeline first.")

In [ ]:
if not mkt.empty and "volume_avg_63d" in mkt.columns:
    vol = mkt["volume_avg_63d"].dropna()
    vol_m = vol / 1e6

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Log-scale histogram
    axes[0].hist(np.log10(vol_m[vol_m > 0] + 1e-3), bins=60,
                 color=PALETTE[4], edgecolor="white")
    axes[0].set_title("Pre-Filing Volume (log₁₀ millions, 63-day avg)")
    axes[0].set_xlabel("log₁₀(Volume, M shares)")
    axes[0].set_ylabel("# Company-years")

    # By sector
    if "sector" in mkt.columns:
        mkt.boxplot(
            column="volume_avg_63d",
            by="sector",
            ax=axes[1],
            vert=False,
            showfliers=False,
            patch_artist=True,
        )
        axes[1].set_xscale("log")
        axes[1].set_title("Pre-Filing Volume by Sector")
        axes[1].set_xlabel("Avg Daily Volume (shares, log scale)")
        plt.suptitle("")

    plt.tight_layout()
    plt.show()

    print("\nVolume percentiles (millions of shares):")
    print((vol_m.describe(percentiles=[.05, .25, .5, .75, .95])).to_frame().T.to_string())

---
## 6  Forward Return Distributions

In [ ]:
if not mkt.empty:
    ret_cols = [c for c in ["fwd_ret_21d", "fwd_ret_63d", "fwd_ret_126d", "fwd_ret_252d"]
                if c in mkt.columns]

    fig, axes = plt.subplots(1, len(ret_cols), figsize=(16, 4), sharey=False)
    labels = {"fwd_ret_21d": "1 month", "fwd_ret_63d": "3 months",
              "fwd_ret_126d": "6 months", "fwd_ret_252d": "12 months"}

    for ax, col in zip(axes, ret_cols):
        data = mkt[col].dropna()
        # Clip at 1st/99th percentile for display
        lo, hi = data.quantile(0.01), data.quantile(0.99)
        data_c = data.clip(lo, hi)
        ax.hist(data_c, bins=80, color=PALETTE[0], edgecolor="none", alpha=0.85)
        ax.axvline(0, color="black", linewidth=1)
        ax.axvline(data.median(), color="#e74c3c", linewidth=1.5,
                   linestyle="--", label=f"median={data.median():.3f}")
        ax.set_title(labels.get(col, col))
        ax.set_xlabel("Log Return")
        ax.legend(fontsize=8)

    axes[0].set_ylabel("# Company-years")
    fig.suptitle("Forward Return Distributions (log-returns, 1%–99% winsorised)",
                 fontsize=13)
    plt.tight_layout()
    plt.show()

    print("\nReturn summary statistics (raw log-returns):")
    print(mkt[ret_cols].describe(percentiles=[.05, .25, .5, .75, .95]).to_string())

---
## 7  Market-Adjusted Return Distributions

In [ ]:
if not mkt.empty:
    adj_cols = [c for c in ["mkt_adj_21d", "mkt_adj_63d", "mkt_adj_126d", "mkt_adj_252d"]
                if c in mkt.columns]
    sec_cols = [c for c in ["sec_adj_21d", "sec_adj_63d", "sec_adj_126d", "sec_adj_252d"]
                if c in mkt.columns]

    if adj_cols:
        fig, axes = plt.subplots(1, len(adj_cols), figsize=(16, 4))
        for ax, col in zip(axes, adj_cols):
            data = mkt[col].dropna()
            lo, hi = data.quantile(0.01), data.quantile(0.99)
            ax.hist(data.clip(lo, hi), bins=80,
                    color=PALETTE[5], edgecolor="none", alpha=0.85,
                    label="mkt-adj")
            sec_col = col.replace("mkt_", "sec_")
            if sec_col in mkt.columns:
                sdata = mkt[sec_col].dropna()
                ax.hist(sdata.clip(lo, hi), bins=80,
                        color=PALETTE[8], edgecolor="none", alpha=0.5,
                        label="sector-adj")
            ax.axvline(0, color="black", linewidth=1)
            ax.set_title(col.replace("mkt_adj_", ""))
            ax.set_xlabel("Excess Log Return")
            ax.legend(fontsize=8)

        axes[0].set_ylabel("# Company-years")
        fig.suptitle("Market- and Sector-Adjusted Returns (1%–99% winsorised)",
                     fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
if not mkt.empty and "mkt_adj_252d" in mkt.columns and "sector" in mkt.columns:
    # Market-adjusted 12m return by sector
    fig, ax = plt.subplots(figsize=(12, 5))
    sectors = mkt.groupby("sector")["mkt_adj_252d"].median().sort_values()
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in sectors]
    ax.barh(sectors.index, sectors.values, color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Median Market-Adjusted 12-Month Return by Sector (FF-12)")
    ax.set_xlabel("Median Excess Log Return")
    plt.tight_layout()
    plt.show()

---
## 8  Stock-Decline Label Preview
*(Threshold: 12-month market-adjusted return < −30%; used for `stock_decline` distress label)*

In [ ]:
if not mkt.empty and "fwd_ret_252d" in mkt.columns:
    DECLINE_THRESHOLD = np.log(1 - 0.30)  # ≈ -0.357 log-return

    mkt["stock_decline"] = (mkt["fwd_ret_252d"] < DECLINE_THRESHOLD).astype("Int8")

    labelled = mkt[mkt["fwd_ret_252d"].notna()]
    decline_rate = labelled["stock_decline"].mean() * 100

    print(f"Stock-decline label (>30% drop in 12 months):")
    print(f"  Threshold (log-return)   : {DECLINE_THRESHOLD:.4f}")
    print(f"  Labelled rows            : {len(labelled):,}")
    print(f"  Positive (decline) rate  : {decline_rate:.2f}%")
    print(f"  Positive count           : {labelled['stock_decline'].sum():,}")

    # By year
    yr_rate = (
        labelled.groupby("filing_year")["stock_decline"]
        .mean()
        .mul(100)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    yr_rate.plot(ax=axes[0], kind="bar", color=PALETTE[3], width=0.7)
    axes[0].set_title("Stock-Decline Event Rate by Filing Year")
    axes[0].set_xlabel("Filing Year")
    axes[0].set_ylabel("Event Rate (%)")
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[0].tick_params(axis='x', rotation=45)

    # By sector
    if "sector" in mkt.columns:
        sec_rate = (
            labelled.groupby("sector")["stock_decline"]
            .mean()
            .sort_values(ascending=True)
            .mul(100)
        )
        sec_rate.plot(ax=axes[1], kind="barh", color=PALETTE[6])
        axes[1].set_title("Stock-Decline Event Rate by Sector")
        axes[1].set_xlabel("Event Rate (%)")
        axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())

    plt.tight_layout()
    plt.show()

---
## 9  Corporate Actions Summary

In [ ]:
if ACTIONS_DIR.exists():
    action_files = sorted(ACTIONS_DIR.glob("*.parquet"))
    print(f"Corporate action files cached: {len(action_files):,}")

    if action_files:
        # Load a sample to check structure
        sample = pd.concat(
            [pd.read_parquet(f).assign(ticker=f.stem) for f in action_files[:500]
             if pd.read_parquet(f).shape[0] > 0],
            ignore_index=True,
        )
        print(f"  Sample rows: {len(sample):,} (from first 500 files with data)")

        if not sample.empty:
            sample.index = pd.to_datetime(sample.index) if not isinstance(sample.index, pd.DatetimeIndex) else sample.index

            splits = sample[sample.get("Stock Splits", pd.Series(0)) != 0] if "Stock Splits" in sample.columns else pd.DataFrame()
            divs   = sample[sample.get("Dividends", pd.Series(0)) != 0] if "Dividends" in sample.columns else pd.DataFrame()

            print(f"  Split events in sample   : {len(splits):,}")
            print(f"  Dividend events in sample: {len(divs):,}")

            if len(splits) > 0 and hasattr(splits.index, 'year'):
                fig, ax = plt.subplots(figsize=(12, 4))
                splits.groupby(splits.index.year).size().plot(
                    kind="bar", ax=ax, color=PALETTE[7]
                )
                ax.set_title("Stock Split Events per Year (sample of 500 tickers)")
                ax.set_xlabel("Year")
                ax.set_ylabel("# Events")
                ax.tick_params(axis='x', rotation=45)
                plt.tight_layout()
                plt.show()
else:
    print("Corporate actions not yet fetched.")
    print("Re-run the pipeline with --with-actions to collect splits/dividends.")

---
## 10  Index / Benchmark Returns

In [ ]:
INDICES_DIR = MARKET_DIR / "indices"

if INDICES_DIR.exists():
    idx_files = sorted(INDICES_DIR.glob("*.parquet"))
    print(f"Index files cached: {len(idx_files)}")

    frames = {}
    for f in idx_files:
        try:
            df = pd.read_parquet(f)
            if not df.empty and "Close" in df.columns:
                label = f.stem.replace("_idx_", "^").replace("_", "/")
                frames[label] = df["Close"]
        except Exception:
            pass

    if frames:
        idx_prices = pd.DataFrame(frames).dropna(how="all")
        # Normalise to 100 at first available date
        idx_norm = idx_prices.div(idx_prices.iloc[0]) * 100

        fig, ax = plt.subplots(figsize=(14, 5))
        for i, col in enumerate(idx_norm.columns):
            idx_norm[col].dropna().plot(
                ax=ax, label=col, linewidth=1.2, color=PALETTE[i % len(PALETTE)]
            )
        ax.set_title("Normalised Index / ETF Returns (base = 100 at first available date)")
        ax.set_xlabel("Date")
        ax.set_ylabel("Indexed Price")
        ax.legend(bbox_to_anchor=(1.01, 1), fontsize=9)
        plt.tight_layout()
        plt.show()

        # Annualised returns table
        ann = {}
        for col in idx_prices.columns:
            s = idx_prices[col].dropna()
            if len(s) > 252:
                log_ret = np.log(s.iloc[-1] / s.iloc[0])
                n_years = len(s) / 252
                ann[col] = {"Ann. Log Return": log_ret / n_years,
                            "Total Return": np.exp(log_ret) - 1,
                            "Years": round(n_years, 1)}
        if ann:
            print("\nAnnualised log returns:")
            print(pd.DataFrame(ann).T.to_string())
else:
    print("Index directory not found — run the pipeline first.")

---
## 11  Data Quality Scorecard

In [ ]:
if not mkt.empty:
    checks = []

    # Universe size
    n_ciks = mkt["cik"].nunique()
    checks.append(("Universe size ≥ 5,000 companies",
                   n_ciks >= 5000, f"{n_ciks:,} CIKs"))

    # 252d return coverage
    cov252 = mkt["fwd_ret_252d"].notna().mean() * 100
    checks.append(("12m return coverage ≥ 70%",
                   cov252 >= 70, f"{cov252:.1f}%"))

    # Market-adj coverage
    if "mkt_adj_252d" in mkt.columns:
        mkt_cov = mkt["mkt_adj_252d"].notna().mean() * 100
        checks.append(("Market-adj 12m coverage ≥ 70%",
                       mkt_cov >= 70, f"{mkt_cov:.1f}%"))

    # Delisting rate reasonable
    del_pct = mkt["delisted"].mean() * 100 if "delisted" in mkt.columns else 0
    checks.append(("Delisting rate < 40%",
                   del_pct < 40, f"{del_pct:.1f}%"))

    # Date range spans study window
    min_yr = mkt["filing_year"].min()
    max_yr = mkt["filing_year"].max()
    checks.append(("Date range covers 2012–2023",
                   min_yr <= 2012 and max_yr >= 2023,
                   f"{min_yr}–{max_yr}"))

    # Sector diversity
    n_sectors = mkt["sector"].nunique() if "sector" in mkt.columns else 0
    checks.append(("Sector diversity ≥ 10 FF-12 sectors",
                   n_sectors >= 10, f"{n_sectors} sectors"))

    # Volume data present
    vol_cov = mkt["volume_avg_63d"].notna().mean() * 100 if "volume_avg_63d" in mkt.columns else 0
    checks.append(("Volume data coverage ≥ 60%",
                   vol_cov >= 60, f"{vol_cov:.1f}%"))

    print("=" * 65)
    print("DATA QUALITY SCORECARD")
    print("=" * 65)
    passed = 0
    for name, ok, value in checks:
        icon = "✓" if ok else "✗"
        print(f"  {icon}  {name:<40}  {value}")
        if ok:
            passed += 1
    print("=" * 65)
    print(f"  Score: {passed}/{len(checks)} checks passed")
    print("=" * 65)

---
## 12  Train / Val / Test Split Preview

In [ ]:
if not mkt.empty:
    # Canonical study-0 splits: train ≤ 2017, val 2018–2019, test 2020–2023
    train_mask = mkt["period_year"] <= 2017
    val_mask   = mkt["period_year"].between(2018, 2019)
    test_mask  = mkt["period_year"] >= 2020

    print("Study-0 time-based splits (on period_end year):")
    print(f"  Train (≤ 2017) : {train_mask.sum():>7,} rows  ({train_mask.mean()*100:.1f}%)")
    print(f"  Val  (2018–19) : {val_mask.sum():>7,} rows  ({val_mask.mean()*100:.1f}%)")
    print(f"  Test (≥ 2020)  : {test_mask.sum():>7,} rows  ({test_mask.mean()*100:.1f}%)")

    if "stock_decline" in mkt.columns:
        print("\nStock-decline event rate per split:")
        for name, mask in [("Train", train_mask), ("Val", val_mask), ("Test", test_mask)]:
            sub = mkt[mask & mkt["fwd_ret_252d"].notna()]
            if len(sub) > 0:
                print(f"  {name:<6}: {sub['stock_decline'].mean()*100:.2f}%  (n={len(sub):,})")